# 01 — Download Data from MAST

**Purpose:** Query MAST for all WFC3/IR FLT exposures from HST proposal 14706 (PI: Glikman),
confirm exactly 6 distinct quasar targets are returned, and download the files into
`data/raw/<quasar>/`.

**Before running:** Read `HANDOFF.md` and `CLAUDE.md` for current project state.

**Outputs:** FLT files in `data/raw/quasar_01/` through `data/raw/quasar_06/`

**Next step:** Run `/rename-quasars` to replace placeholder folder names with real target names,
then open `02_inspect_raw.ipynb`.

---
**IMPORTANT — Query parameters:**
- `project='HST'` (not `'HAP'`) — HAP returns processed mosaics, not individual exposures
- `productSubGroupDescription='FLT'` — calibrated individual exposures from CALWF3
- `proposal_id='14706'`
- `pi_last_name='Glikman'`
- `instrument_name='WFC3/IR'`

## Imports

In [ ]:
import os
import shutil
import glob
from pathlib import Path
from astroquery.mast import Observations

## Define paths

In [ ]:
RAW_DIR = Path('../data/raw')

## Query MAST

In [ ]:
obs_table = Observations.query_criteria(
    proposal_id='14706',
    pi_last_name='Glikman',
    instrument_name='WFC3/IR',
    project='HST'
)

print(f'Observations found: {len(obs_table)}')
print(obs_table['target_name', 'obs_id', 'filters', 't_exptime'])

## Confirm 6 distinct targets

In [ ]:
targets = list(set(obs_table['target_name']))
print(f'Distinct targets: {len(targets)}')
for t in sorted(targets):
    print(f'  {t}')

assert len(targets) == 6, f'Expected 6 targets, got {len(targets)} — review query before downloading'

## Get product list and filter to FLT files

In [ ]:
products = Observations.get_product_list(obs_table)

filtered = Observations.filter_products(
    products,
    productSubGroupDescription='FLT',
    project='CALWF3'
)

print(f'FLT files to download: {len(filtered)}')
filtered['target_name', 'productFilename', 'size']

## Download files

In [ ]:
Observations.download_products(filtered, cache=True)

## Move files into per-quasar directories

In [ ]:
# Move downloaded files into data/raw/quasar_XX/ based on target_name
# TODO: populate this mapping after confirming target names above
# Example: target_name_to_folder = {'J0100+2802': 'quasar_01', ...}
target_name_to_folder = {}

for flt in glob.glob('./mastDownload/HST/*/*flt.fits'):
    # placeholder — update once target_name_to_folder is populated
    flt_path = Path(flt)
    print(f'Found: {flt_path.name}')

# shutil.rmtree('mastDownload/')  # uncomment after verifying all files moved correctly

## Next steps

1. Run `/rename-quasars` to replace placeholder folder names with real target names
2. Run `/validate-downloads` to confirm all files are present and uncorrupted
3. Open `02_inspect_raw.ipynb`